## Imports and Config

In [ ]:
%load_ext autoreload
%autoreload 2

Custom imports from the `config/settings.py` file.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [ ]:
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

from haystack import Document

In [ ]:
from sem_rag.config.settings import SEMDataset, Evaluation
from sem_rag.retrieval.gabor import GaborDescriptor
from sem_rag.embedding.embedder import CLIPEmbedder, QwenEmbedder
from sem_rag.retrieval.components import MMRReranker
from sem_rag.image.descriptions import get_crop_path

In [ ]:
EVAL_DEPTH = 50
MMR_CANDIDATES = 50
TOP_K = 5
NEAR_DUPLICATE_THRESHOLD = 0.90
SESSION_WINDOW = 20  # IDs within ±20 of the query are treated as the same session
CHUNK_SIZE = 1000

# 1. Category Retrieval

The user uploads an SEM image and asks for similar ones. Like text part 1, an image is considered **relevant if it has the same human-assigned category as the query image** (`SEMDataset.CATEGORIES`, Aversa et al.). No judge is needed.

Protocol (leave-one-out, the standard query-by-example protocol in content-based image retrieval):
- **Every image in the collection is used once as the query** (14,169 queries). It is removed from its own ranking.
- All rankings are computed locally from the stored embeddings, which are identical to what the production `QueryEmbedder` returns for an uploaded image (checked: cosine similarity 1.0). Qdrant's approximate search can differ marginally from this exact search.
- Three settings, from what a user sees to the strictest test of category understanding:
  - **all**: every other image can be retrieved. This is what a user sees.
  - **dedup**: near-duplicates of the query (DINO cosine similarity ≥ `NEAR_DUPLICATE_THRESHOLD`) are also removed. About 45% of images have one: they are usually the same sample slightly shifted, so retrieving them gives the right category "for free".
  - **session**: images whose ID is within ±`SESSION_WINDOW` of the query are also removed. IDs are sequential and categories come in contiguous runs of IDs (~400 runs, ~35 images each), so neighbouring IDs are very likely the same acquisition session or sample. Without this, a system can score well by recognising the session (same sample, detector, contrast) rather than the morphology.

Systems:
- **Gabor**: hand-crafted texture features (24-dim), Euclidean distance. Reported as stored (`raw`) and after standardising each feature (`z`), since the raw distance is dominated by the features with the largest scale.
- **CLIP** (ViT-B/32) and **Qwen3-VL** image embeddings, cosine similarity.
- **InternRAG**: Qwen3-VL top `MMR_CANDIDATES` followed by MMR, as in production (only `TOP_K` images, so evaluated at k=5 only).
- **DINOv3**: a strong vision-only model. In the `dedup` and `session` settings it is disadvantaged, since the removed near-duplicates are by definition its own top neighbours.

Limitations: relevance is coarse (any image of the right category counts), labels are somewhat noisy, and images from the same sample are not independent, so the confidence intervals are optimistic.

## Setup

In [ ]:
IMAGE_RESULTS_DIR = PROJECT_ROOT / "data" / "evaluation" / "image_category_retrieval"
IMAGE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
labels = pd.read_csv(SEMDataset.CATEGORIES, dtype={"TIF_SEMImageID": str})

IMAGE_IDS = labels["TIF_SEMImageID"].tolist()
CATEGORY_OF = dict(zip(IMAGE_IDS, labels["mapped_category"]))
CATEGORIES = sorted(labels["mapped_category"].unique())
N_IMAGES = len(IMAGE_IDS)

# Integer category code per image, aligned with IMAGE_IDS
category_codes = labels["mapped_category"].map({category: i for i, category in enumerate(CATEGORIES)}).to_numpy()
numeric_ids = np.asarray(IMAGE_IDS, dtype=int)

print(f"{N_IMAGES} images, {len(CATEGORIES)} categories")

### Embeddings

All matrices are aligned with `IMAGE_IDS` (row `i` is image `IMAGE_IDS[i]`) and L2-normalised, so cosine similarity is a dot product.

In [ ]:
def load_pt_embeddings(directory: Path) -> np.ndarray:
    embeddings = np.vstack([torch.load(directory / f"{image_id}.pt").numpy() for image_id in IMAGE_IDS]).astype(np.float32)
    return embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

In [ ]:
qwen_embeddings = load_pt_embeddings(SEMDataset.EMBEDDINGS_QWEN)
clip_embeddings = load_pt_embeddings(SEMDataset.EMBEDDINGS_CLIP)

dino_files = {file.name.split("_")[0]: file for file in SEMDataset.EMBEDDINGS_DINO.glob("*.npz")}
dino_embeddings = np.vstack([np.load(dino_files[image_id])["cls"] for image_id in IMAGE_IDS]).astype(np.float32)
dino_embeddings /= np.linalg.norm(dino_embeddings, axis=1, keepdims=True)

gabor = np.load(Evaluation.GABOR_FILE)
gabor_index = {str(image_id): i for i, image_id in enumerate(gabor["image_ids"])}
gabor_raw = gabor["features"][[gabor_index[image_id] for image_id in IMAGE_IDS]].astype(np.float32)
gabor_z = (gabor_raw - gabor_raw.mean(axis=0)) / gabor_raw.std(axis=0)

print("Qwen", qwen_embeddings.shape, "| CLIP", clip_embeddings.shape, "| DINO", dino_embeddings.shape, "| Gabor", gabor_raw.shape)

## Retrieval

For each chunk of queries, every system scores the whole collection. The query itself (and, in the `dedup` setting, its near-duplicates) gets score `-inf`, then the top `EVAL_DEPTH` images are kept.

In [ ]:
def cosine_scores(embeddings: np.ndarray):
    return lambda rows: embeddings[rows] @ embeddings.T


def euclidean_scores(features: np.ndarray):
    squared_norms = (features ** 2).sum(axis=1)
    # Negative squared distance: same ranking as Euclidean distance, higher is better
    return lambda rows: -(squared_norms[rows, None] - 2 * features[rows] @ features.T + squared_norms[None, :])


SCORERS = {
    "Gabor (raw)": euclidean_scores(gabor_raw),
    "Gabor (z)": euclidean_scores(gabor_z),
    "CLIP": cosine_scores(clip_embeddings),
    "Qwen3-VL": cosine_scores(qwen_embeddings),
    "DINOv3": cosine_scores(dino_embeddings),
}
SETTINGS = ["all", "dedup", "session"]

In [ ]:
def top_k_indices(scores: np.ndarray, k: int) -> np.ndarray:
    """Indices of the k highest scores per row, sorted by decreasing score."""
    top = np.argpartition(-scores, k, axis=1)[:, :k]
    order = np.argsort(-np.take_along_axis(scores, top, axis=1), axis=1)
    return np.take_along_axis(top, order, axis=1)

In [ ]:
rankings = {setting: {system: np.zeros((N_IMAGES, EVAL_DEPTH), dtype=np.int32) for system in SCORERS} for setting in SETTINGS}
n_relevant = {setting: np.zeros(N_IMAGES, dtype=np.int32) for setting in SETTINGS}    # same-category images that can be retrieved
n_candidates = {setting: np.zeros(N_IMAGES, dtype=np.int32) for setting in SETTINGS}  # images that can be retrieved
n_near_duplicates = np.zeros(N_IMAGES, dtype=np.int32)

for start in range(0, N_IMAGES, CHUNK_SIZE):
    rows = np.arange(start, min(start + CHUNK_SIZE, N_IMAGES))

    excluded = {"all": np.zeros((len(rows), N_IMAGES), dtype=bool)}
    excluded["all"][np.arange(len(rows)), rows] = True
    excluded["dedup"] = excluded["all"] | (dino_embeddings[rows] @ dino_embeddings.T >= NEAR_DUPLICATE_THRESHOLD)
    excluded["session"] = excluded["dedup"] | (np.abs(numeric_ids[rows, None] - numeric_ids[None, :]) <= SESSION_WINDOW)
    n_near_duplicates[rows] = excluded["dedup"].sum(axis=1) - 1

    same_category = category_codes[rows, None] == category_codes[None, :]
    for setting in SETTINGS:
        n_relevant[setting][rows] = (same_category & ~excluded[setting]).sum(axis=1)
        n_candidates[setting][rows] = (~excluded[setting]).sum(axis=1)

    for system, scorer in SCORERS.items():
        scores = scorer(rows)
        for setting in SETTINGS:
            rankings[setting][system][rows] = top_k_indices(np.where(excluded[setting], -np.inf, scores), EVAL_DEPTH)

print(f"Images with at least one near-duplicate: {(n_near_duplicates > 0).mean():.1%}")

In [ ]:
def mmr_select(query_embedding: np.ndarray, candidates: np.ndarray, top_k: int = TOP_K, lambda_: float = 0.8) -> np.ndarray:
    """Same greedy selection as MMRReranker.run, on precomputed normalised embeddings."""
    embeddings = qwen_embeddings[candidates]
    query_similarities = embeddings @ query_embedding
    candidate_similarities = embeddings @ embeddings.T

    selected = [int(np.argmax(query_similarities))]
    remaining = [i for i in range(len(candidates)) if i != selected[0]]

    while len(selected) < top_k and remaining:
        mmr_scores = [(lambda_ * query_similarities[i] - (1 - lambda_) * np.max(candidate_similarities[i, selected]), i) for i in remaining]

        _, best = max(mmr_scores)
        selected.append(best)
        remaining.remove(best)

    return candidates[selected]


for setting in SETTINGS:
    candidates = rankings[setting]["Qwen3-VL"][:, :MMR_CANDIDATES]
    rankings[setting]["InternRAG"] = np.vstack([mmr_select(qwen_embeddings[query], candidates[query]) for query in range(N_IMAGES)])

In [ ]:
# Check against the production component on a random sample of queries
mmr = MMRReranker(top_k=TOP_K, lambda_=0.8)
rng = np.random.default_rng(0)

for query in rng.choice(N_IMAGES, 50, replace=False):
    candidates = rankings["all"]["Qwen3-VL"][query, :MMR_CANDIDATES]
    documents = [Document(id=IMAGE_IDS[i], embedding=qwen_embeddings[i].tolist()) for i in candidates]
    selected = mmr.run(query_embedding=qwen_embeddings[query].tolist(), documents=documents)["documents"]
    assert [document.id for document in selected] == [IMAGE_IDS[i] for i in rankings["all"]["InternRAG"][query]]

print("NumPy MMR matches MMRReranker")

## Metrics

Same definitions as text part 1, computed on all queries at once:
- **P@k**: fraction of the top-k images that are relevant.
- **nDCG@10**: like P@10, but relevant images ranked higher count more.
- **AP@50**: average precision over the top 50, normalised by `min(#relevant, 50)`.
- **Random**: expected score of a random ranking, i.e. the share of retrievable images that have the query's category.

Categories are very imbalanced (MEMS ~25%, Fibres ~1%), so a plain average over all images mostly measures MEMS and Patterned_surface. The headline number is the **macro average**: the mean over images within each category, then the mean over the 10 categories (each category counts the same, like the 5 queries per category of text part 1). The micro average over all images is reported too.

In [ ]:
SYSTEMS = ["Random", "Gabor (raw)", "Gabor (z)", "CLIP", "Qwen3-VL", "InternRAG", "DINOv3"]
METRICS = ["P@5", "P@10", "nDCG@10", "AP@50"]

DISCOUNTS = 1 / np.log2(np.arange(2, EVAL_DEPTH + 2))


def metrics_for(ranking: np.ndarray, relevant_count: np.ndarray) -> dict[str, np.ndarray]:
    relevant = category_codes[ranking] == category_codes[:, None]
    depth = relevant.shape[1]
    nan = np.full(len(relevant), np.nan)

    hits = np.cumsum(relevant, axis=1)
    ideal_dcg = np.cumsum(DISCOUNTS[:10])[np.minimum(relevant_count, 10) - 1]
    average_precision = (relevant * hits / np.arange(1, depth + 1)).sum(axis=1) / np.minimum(relevant_count, EVAL_DEPTH)

    return {
        "P@5": relevant[:, :5].mean(axis=1),
        "P@10": relevant[:, :10].mean(axis=1) if depth >= 10 else nan,
        "nDCG@10": (relevant[:, :10] @ DISCOUNTS[:10]) / ideal_dcg if depth >= 10 else nan,
        "AP@50": average_precision if depth >= EVAL_DEPTH else nan,
    }


frames = []

for setting in SETTINGS:
    base = pd.DataFrame({
        "image_id": IMAGE_IDS,
        "category": labels["mapped_category"],
        "near_duplicates": n_near_duplicates,
        "setting": setting,
    })

    prior = n_relevant[setting] / n_candidates[setting]
    frames.append(base.assign(system="Random", **{metric: prior for metric in METRICS}))

    for system in SYSTEMS[1:]:
        frames.append(base.assign(system=system, **metrics_for(rankings[setting][system], n_relevant[setting])))

image_results = pd.concat(frames, ignore_index=True)
image_results.head()

## Results

Macro average with a 95% bootstrap confidence interval. Queries are resampled **within each category** (stratified bootstrap), so each resample keeps the category sizes and the macro average stays balanced.

In [ ]:
def macro_mean(results: pd.DataFrame, metric: str) -> float:
    return results.groupby("category")[metric].mean().mean()


def stratified_bootstrap_ci(results: pd.DataFrame, metric: str, n_resamples: int = 1_000, seed: int = 0) -> tuple[float, float]:
    """95% CI of the macro average, resampling queries within each category."""
    rng = np.random.default_rng(seed)
    groups = [group[metric].to_numpy() for _, group in results.groupby("category")]
    means = np.mean([
        [rng.choice(values, len(values)).mean() for values in groups]
        for _ in range(n_resamples)
    ], axis=1)
    return np.quantile(means, 0.025), np.quantile(means, 0.975)


def summary_table(setting: str) -> pd.DataFrame:
    summary = {}
    for system in SYSTEMS:
        results = image_results[(image_results["setting"] == setting) & (image_results["system"] == system)]
        summary[system] = {}
        for metric in METRICS:
            if results[metric].isna().all():
                summary[system][metric] = "-"
                continue
            low, high = stratified_bootstrap_ci(results, metric)
            summary[system][metric] = f"{macro_mean(results, metric):.3f} [{low:.3f}, {high:.3f}]"
    return pd.DataFrame(summary).T

### Setting `all` (macro average)

In [ ]:
summary_table("all")

### Setting `dedup` (macro average)

In [ ]:
summary_table("dedup")

### Setting `session` (macro average)

In [ ]:
summary_table("session")

Micro average (all 14,169 queries weighted equally, so dominated by the large categories):

In [ ]:
(
    image_results
    .groupby(["system", "setting"])[METRICS]
    .mean()
    .unstack("setting")
    .reindex(SYSTEMS)
    .round(3)
)

P@5 by category, in the `all` and `session` settings. Compare each category with its own **Random** column: the same P@5 means much more for Fibres (~1% of the collection) than for MEMS (~25%).

In [ ]:
(
    image_results[image_results["setting"].isin(["all", "session"])]
    .pivot_table(index="category", columns=["setting", "system"], values="P@5", aggfunc="mean")
    .reindex(columns=SYSTEMS, level="system")
    .round(2)
)

### Effect of near-duplicates

P@5 of Qwen3-VL in the `all` setting, split by how many near-duplicates the query image has. If the score is much higher for images with many near-duplicates, part of the `all` result comes from same-scene matches rather than category understanding.

In [ ]:
qwen_all = image_results[(image_results["setting"] == "all") & (image_results["system"].isin(["CLIP", "Qwen3-VL", "InternRAG"]))].copy()
qwen_all["near_duplicate_group"] = pd.cut(qwen_all["near_duplicates"], bins=[-1, 0, 4, 19, np.inf], labels=["0", "1-4", "5-19", "20+"])

qwen_all.pivot_table(index="near_duplicate_group", columns="system", values="P@5", aggfunc="mean", observed=True).assign(
    queries=qwen_all[qwen_all["system"] == "Qwen3-VL"].groupby("near_duplicate_group", observed=True).size()
).round(3)

### Effect of the acquisition session

Share of top-5 results whose ID is within ±5, ±20 or ±100 of the query (i.e. likely the same session), per setting. A random ranking gives about 0.1%, 0.2% and 0.7%.

In [ ]:
session_share = {}

for setting in SETTINGS:
    for system in SYSTEMS[1:]:
        distances = np.abs(numeric_ids[rankings[setting][system][:, :TOP_K]] - numeric_ids[:, None])
        session_share[(setting, system)] = {f"±{window}": (distances <= window).mean() for window in [5, 20, 100]}

pd.DataFrame(session_share).T.unstack(0).round(2)

## Qualitative Check

Top-5 images of each system for one query image. Green titles are relevant, red titles are not (with the actual category).

In [ ]:
def show_query(image_id: str, setting: str = "dedup", k: int = TOP_K):
    query = IMAGE_IDS.index(image_id)
    systems = SYSTEMS[1:]

    fig, axes = plt.subplots(len(systems), k + 1, figsize=(2.2 * (k + 1), 2.4 * len(systems)))
    fig.suptitle(f"Query {image_id} ({CATEGORY_OF[image_id]}), setting: {setting}")

    for ax in axes.flat:
        ax.axis("off")

    axes[0, 0].imshow(np.load(get_crop_path(image_id)), cmap="gray")
    axes[0, 0].set_title(f"QUERY\n{image_id}", fontsize=7)

    for row, system in enumerate(systems):
        for col, index in enumerate(rankings[setting][system][query, :k], start=1):
            retrieved_id = IMAGE_IDS[index]
            category = CATEGORY_OF[retrieved_id]
            ax = axes[row, col]
            ax.imshow(np.load(get_crop_path(retrieved_id)), cmap="gray")
            ax.set_title(
                f"{system}\n{retrieved_id} {category}" if col == 1 else f"\n{retrieved_id} {category}",
                fontsize=7,
                color="green" if category == CATEGORY_OF[image_id] else "red",
            )

    plt.tight_layout()
    plt.show()


show_query("38")

Save the per-query results and the rankings (as image IDs) so the numbers can be reproduced without recomputing.

In [ ]:
image_results.to_csv(IMAGE_RESULTS_DIR / "results.csv", index=False)

np.savez_compressed(
    IMAGE_RESULTS_DIR / "rankings.npz",
    image_ids=np.asarray(IMAGE_IDS),
    **{f"{setting}/{system}": ranking for setting in SETTINGS for system, ranking in rankings[setting].items()},
)

# 2. Robustness: Finding a Modified Image

The image analogue of text target retrieval (known-item search): the query is an indexed image that has been **modified** in a realistic way, and the system should still return the original. Each modification simulates "the same sample, imaged slightly differently" by the user (other field of view, orientation, detector settings, scan speed or resolution). This follows copy-detection and near-duplicate benchmarks such as INRIA Holidays (Jégou et al., 2008) and DISC21 (Douze et al., 2021).

Protocol:
1. Sample `TARGETS_PER_CATEGORY` target images per category (part 1's labels).
2. Apply each modification in `TRANSFORMS` to every target. `identity` (unmodified image) is a sanity check: every system should find the target at rank 1.
3. Embed the modified image with each system, exactly as an uploaded image would be embedded, and rank the whole collection (the target included).

Metrics, as in text target retrieval:
- **R@k**: fraction of queries whose target is in the top k.
- **MRR**: mean of 1/rank of the target (0 if not within `ROBUSTNESS_DEPTH`).
- **Strict** counts only the target. **Soft** also accepts a near-duplicate of the target (DINO cosine similarity ≥ `NEAR_DUPLICATE_THRESHOLD`): after zooming, a shifted crop of the same scene can legitimately be closer than the original.

What it tells you: which systems match the *content* of the image and which match surface statistics (brightness, noise, scale). Part 1 measures whether results are relevant; this part measures whether they are stable under the variations a user's own image will have.

## Targets and Modifications

In [ ]:
TARGETS_PER_CATEGORY = 30
ROBUSTNESS_DEPTH = 100
ROBUSTNESS_KS = [1, 5, 10]

ROBUSTNESS_DIR = PROJECT_ROOT / "data" / "evaluation" / "image_robustness"
MODIFIED_DIR = ROBUSTNESS_DIR / "queries"
MODIFIED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
robustness_targets = (
    labels.rename(columns={"TIF_SEMImageID": "image_id", "mapped_category": "category"})
    .sample(frac=1, random_state=0)
    .groupby("category")
    .head(TARGETS_PER_CATEGORY)
    .sort_values(["category", "image_id"])
    .reset_index(drop=True)
)
ROBUSTNESS_IDS = robustness_targets["image_id"].tolist()

print(f"{len(ROBUSTNESS_IDS)} targets")

Each modification is deterministic (seeded by the image ID), so the queries are reproducible:
- `identity`: unmodified (sanity check).
- `zoom`: a random window covering 70% of the width and height, resized back to 512×512 (different field of view / magnification).
- `rotate`: rotated by 90° (different sample orientation).
- `contrast`: gamma 0.6 and contrast reduced by 30% (different brightness/contrast settings).
- `noise`: Gaussian noise with σ = 20 grey levels (faster, noisier scan).
- `low_res`: downscaled to 128×128 and back (lower resolution).

In [ ]:
SIZE = 512


def to_uint8(image: np.ndarray) -> np.ndarray:
    return np.clip(np.rint(image), 0, 255).astype(np.uint8)


def resize(image: np.ndarray, size: int) -> np.ndarray:
    return np.asarray(Image.fromarray(image, mode="L").resize((size, size), Image.Resampling.BICUBIC))


def zoom(image: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    window = int(0.7 * SIZE)
    top, left = rng.integers(0, SIZE - window + 1, size=2)
    return resize(image[top:top + window, left:left + window], SIZE)


def contrast(image: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    gamma = 255 * (image / 255) ** 0.6
    return to_uint8(gamma.mean() + 0.7 * (gamma - gamma.mean()))


TRANSFORMS = {
    "identity": lambda image, rng: image,
    "zoom": zoom,
    "rotate": lambda image, rng: np.rot90(image).copy(),
    "contrast": contrast,
    "noise": lambda image, rng: to_uint8(image + rng.normal(0, 20, image.shape)),
    "low_res": lambda image, rng: resize(resize(image, 128), SIZE),
}


def modified_path(image_id: str, transform: str) -> Path:
    return MODIFIED_DIR / f"{image_id}_{transform}.npy"


for image_id in ROBUSTNESS_IDS:
    image = np.load(get_crop_path(image_id))
    for transform, function in TRANSFORMS.items():
        path = modified_path(image_id, transform)
        if not path.exists():
            np.save(path, function(image, np.random.default_rng(int(image_id))))

In [ ]:
def show_modifications(image_id: str):
    fig, axes = plt.subplots(1, len(TRANSFORMS), figsize=(2.4 * len(TRANSFORMS), 2.8))
    fig.suptitle(f"{image_id} ({CATEGORY_OF[image_id]})")
    for ax, transform in zip(axes, TRANSFORMS):
        ax.imshow(np.load(modified_path(image_id, transform)), cmap="gray", vmin=0, vmax=255)
        ax.set_title(transform, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


show_modifications(ROBUSTNESS_IDS[0])

## Query Embeddings

Modified images are embedded with the same code that produced the collection embeddings (`QwenEmbedder`, `CLIPEmbedder`, `GaborDescriptor`). Qwen3-VL calls the vLLM endpoint (one call per query, run in parallel), so the embeddings are cached in `query_embeddings.npz` and only missing transforms are computed.

In [ ]:
qwen_image_embedder = QwenEmbedder()
clip_image_embedder = CLIPEmbedder()
gabor_descriptor = GaborDescriptor()

gabor_mean, gabor_std = gabor_raw.mean(axis=0), gabor_raw.std(axis=0)


def normalise(embeddings: list) -> np.ndarray:
    embeddings = np.asarray(embeddings, dtype=np.float32)
    return embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)


def embed_transform(transform: str) -> dict[str, np.ndarray]:
    paths = [modified_path(image_id, transform) for image_id in ROBUSTNESS_IDS]
    with ThreadPoolExecutor(max_workers=8) as executor:
        qwen = list(executor.map(qwen_image_embedder.embed_image, paths))
    gabor_features = np.vstack([gabor_descriptor.extract(np.load(path)) for path in paths])
    return {
        "Qwen3-VL": normalise(qwen),
        "CLIP": normalise([clip_image_embedder.embed_image(path) for path in paths]),
        "Gabor (raw)": gabor_features,
        "Gabor (z)": (gabor_features - gabor_mean) / gabor_std,
    }


cache_path = ROBUSTNESS_DIR / "query_embeddings.npz"
query_embeddings = dict(np.load(cache_path)) if cache_path.exists() else {}

for transform in TRANSFORMS:
    if f"Qwen3-VL/{transform}" in query_embeddings:
        continue
    for system, embeddings in embed_transform(transform).items():
        query_embeddings[f"{system}/{transform}"] = embeddings
    np.savez_compressed(cache_path, **query_embeddings)
    print(f"Embedded {transform}")

## Retrieval

Every system ranks the whole collection (the target included) and keeps the top `ROBUSTNESS_DEPTH`. InternRAG is MMR over the Qwen3-VL top `MMR_CANDIDATES`, with the same `mmr_select` as part 1.

In [ ]:
COLLECTION = {
    "Gabor (raw)": gabor_raw,
    "Gabor (z)": gabor_z,
    "CLIP": clip_embeddings,
    "Qwen3-VL": qwen_embeddings,
}


def collection_scores(system: str, queries: np.ndarray) -> np.ndarray:
    collection = COLLECTION[system]
    if system.startswith("Gabor"):
        # Negative squared Euclidean distance: same ranking as GaborRetriever, higher is better
        return -((queries ** 2).sum(axis=1)[:, None] - 2 * queries @ collection.T + (collection ** 2).sum(axis=1)[None, :])
    return queries @ collection.T


robustness_rankings = {}

for transform in TRANSFORMS:
    for system in COLLECTION:
        queries = query_embeddings[f"{system}/{transform}"]
        robustness_rankings[(transform, system)] = top_k_indices(collection_scores(system, queries), ROBUSTNESS_DEPTH)

    qwen_queries = query_embeddings[f"Qwen3-VL/{transform}"]
    candidates = robustness_rankings[(transform, "Qwen3-VL")][:, :MMR_CANDIDATES]
    robustness_rankings[(transform, "InternRAG")] = np.vstack([
        mmr_select(qwen_queries[i], candidates[i]) for i in range(len(ROBUSTNESS_IDS))
    ])

## Metrics

In [ ]:
image_index = {image_id: i for i, image_id in enumerate(IMAGE_IDS)}
target_rows = np.array([image_index[image_id] for image_id in ROBUSTNESS_IDS])

# Collection indices of each target's near-duplicates (the target itself excluded)
target_similarities = dino_embeddings[target_rows] @ dino_embeddings.T
TARGET_NEAR_DUPLICATES = [
    set(np.where(target_similarities[i] >= NEAR_DUPLICATE_THRESHOLD)[0]) - {row}
    for i, row in enumerate(target_rows)
]


def first_rank(ranking: np.ndarray, hits: set[int]) -> int | None:
    """1-based rank of the first index in `hits`, or None if none was retrieved."""
    for rank, index in enumerate(ranking, start=1):
        if index in hits:
            return rank
    return None


rows = []

for (transform, system), ranking in robustness_rankings.items():
    for i, image_id in enumerate(ROBUSTNESS_IDS):
        for match, hits in [("strict", {target_rows[i]}), ("soft", {target_rows[i]} | TARGET_NEAR_DUPLICATES[i])]:
            rank = first_rank(ranking[i], hits)
            row = {
                "image_id": image_id,
                "category": CATEGORY_OF[image_id],
                "transform": transform,
                "system": system,
                "match": match,
                "rank": rank,
                "MRR": np.nan if system == "InternRAG" else (1 / rank if rank else 0.0),
            }
            for k in ROBUSTNESS_KS:
                row[f"R@{k}"] = float(rank is not None and rank <= k) if ranking.shape[1] >= k else np.nan
            rows.append(row)

robustness_results = pd.DataFrame(rows)

ROBUSTNESS_SYSTEMS = ["Gabor (raw)", "Gabor (z)", "CLIP", "Qwen3-VL", "InternRAG"]
ROBUSTNESS_METRICS = [f"R@{k}" for k in ROBUSTNESS_KS] + ["MRR"]

## Results

The `identity` row must be 1.0 everywhere (InternRAG: R@1 and R@5). Anything lower means the query is not embedded like the collection.

In [ ]:
def robustness_table(metric: str, match: str = "strict") -> pd.DataFrame:
    return (
        robustness_results[robustness_results["match"] == match]
        .pivot_table(index="transform", columns="system", values=metric, aggfunc="mean")
        .reindex(index=list(TRANSFORMS), columns=ROBUSTNESS_SYSTEMS)
        .round(3)
    )


print("Strict R@1")
display(robustness_table("R@1"))
print("Strict R@10 (InternRAG: R@5)")
display(robustness_table("R@10").assign(InternRAG=robustness_table("R@5")["InternRAG"]))

Overall (all modifications except `identity`), strict and soft, with 95% bootstrap confidence intervals over targets:

In [ ]:
def bootstrap_ci(values, n_resamples: int = 10_000, seed: int = 0) -> tuple[float, float]:
    """95% bootstrap confidence interval of the mean."""
    values = np.asarray(values.dropna())
    rng = np.random.default_rng(seed)
    means = rng.choice(values, size=(n_resamples, len(values))).mean(axis=1)
    return np.quantile(means, 0.025), np.quantile(means, 0.975)


summary = {}
modified =robustness_results[robustness_results["transform"] != "identity"]

for system in ROBUSTNESS_SYSTEMS:
    for match in ["strict", "soft"]:
        results = modified[(modified["system"] == system) & (modified["match"] == match)]
        summary[(system, match)] = {}
        for metric in ROBUSTNESS_METRICS:
            # Resample targets (all modifications of a target together), since they share the same image
            per_target = results.groupby("image_id")[metric].mean()
            if per_target.isna().all():
                summary[(system, match)][metric] = "-"
                continue
            low, high = bootstrap_ci(per_target)
            summary[(system, match)][metric] = f"{per_target.mean():.3f} [{low:.3f}, {high:.3f}]"

pd.DataFrame(summary).T

Strict R@10 by category (all modifications except `identity`):

In [ ]:
(
    modified[modified["match"] == "strict"]
    .pivot_table(index="category", columns="system", values="R@10", aggfunc="mean")
    [ROBUSTNESS_SYSTEMS[:-1]]
    .round(2)
)

## Qualitative Check

The modified query and each system's top 5. Green: the target; orange: a near-duplicate of the target; black: anything else.

In [ ]:
def show_robustness(image_id: str, transform: str, k: int = TOP_K):
    i = ROBUSTNESS_IDS.index(image_id)

    fig, axes = plt.subplots(len(ROBUSTNESS_SYSTEMS), k + 1, figsize=(2.2 * (k + 1), 2.4 * len(ROBUSTNESS_SYSTEMS)))
    fig.suptitle(f"Target {image_id} ({CATEGORY_OF[image_id]}), modification: {transform}")

    for ax in axes.flat:
        ax.axis("off")

    axes[0, 0].imshow(np.load(modified_path(image_id, transform)), cmap="gray", vmin=0, vmax=255)
    axes[0, 0].set_title("QUERY", fontsize=7)

    for row, system in enumerate(ROBUSTNESS_SYSTEMS):
        for col, index in enumerate(robustness_rankings[(transform, system)][i, :k], start=1):
            if index == target_rows[i]:
                color = "green"
            elif index in TARGET_NEAR_DUPLICATES[i]:
                color = "orange"
            else:
                color = "black"
            ax = axes[row, col]
            ax.imshow(np.load(get_crop_path(IMAGE_IDS[index])), cmap="gray")
            ax.set_title(f"{system}\n{IMAGE_IDS[index]}" if col == 1 else f"\n{IMAGE_IDS[index]}", fontsize=7, color=color)

    plt.tight_layout()
    plt.show()


show_robustness(ROBUSTNESS_IDS[0], "zoom")

Save the per-query results so the numbers can be reproduced without re-embedding (the modified images and query embeddings are already cached in `ROBUSTNESS_DIR`).

In [ ]:
robustness_results.to_csv(ROBUSTNESS_DIR / "results.csv", index=False)